# Step 1：資料前處理（精簡版）
- 只讀取 scan 0–900（2026/01/27 當日資料）
- 清理資料
- 7/3 時序切分
- 標準化
- 儲存 train_final.parquet / test_final.parquet / scaler.pkl


In [1]:
import os
import glob
import pickle
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

# ============================================================
# 設定區
# ============================================================
DATA_DIR    = r"C:\Users\peggy\Desktop\深度學習\aligned strict"
OUTPUT_DIR  = r"C:\Users\peggy\Desktop\深度學習\processed_data"
TRAIN_RATIO = 0.7
CHUNK_SIZE  = 30        # 每次處理 30 個檔案
SCAN_MIN    = 0         # scan_id 下限
SCAN_MAX    = 900       # scan_id 上限

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"資料來源：{DATA_DIR}")
print(f"輸出資料夾：{OUTPUT_DIR}")
print(f"讀取範圍：scan {SCAN_MIN} ~ {SCAN_MAX}")


資料來源：C:\Users\peggy\Desktop\深度學習\aligned strict
輸出資料夾：C:\Users\peggy\Desktop\深度學習\processed_data
讀取範圍：scan 0 ~ 900


In [2]:
# ============================================================
# 欄位定義
# ============================================================
KEEP_COLS = [
    "scan_id", "wavelength_nm", "IL_Min", "x_pos", "y_pos",
    "N0160_acc_x", "N0160_acc_y", "N0160_acc_z", "N0160_acc_mag",
    "N0160_gyr_x", "N0160_gyr_y", "N0160_gyr_z",
    "N0180_acc_x", "N0180_acc_y", "N0180_acc_z", "N0180_acc_mag",
    "N0180_gyr_x", "N0180_gyr_y", "N0180_gyr_z",
    "N0294_acc_x", "N0294_acc_y", "N0294_acc_z", "N0294_acc_mag",
    "N0294_gyr_x", "N0294_gyr_y", "N0294_gyr_z",
    "N1157_acc_x", "N1157_acc_y", "N1157_acc_z", "N1157_acc_mag",
    "N1157_gyr_x", "N1157_gyr_y", "N1157_gyr_z",
    "N1159_acc_x", "N1159_acc_y", "N1159_acc_z", "N1159_acc_mag",
    "N1159_gyr_x", "N1159_gyr_y", "N1159_gyr_z",
    "U0042_acc_x", "U0042_acc_y", "U0042_acc_z", "U0042_acc_mag",
    "U0044_acc_x", "U0044_acc_y", "U0044_acc_z", "U0044_acc_mag",
]

FEATURE_COLS = [c for c in KEEP_COLS if c not in ["scan_id", "IL_Min"]]
TARGET_COL   = "IL_Min"
VIB_FEATURES = [c for c in FEATURE_COLS
                if any(s in c for s in
                       ["N0160","N0180","N0294","N1157","N1159","U0042","U0044"])]

print(f"特徵數：{len(FEATURE_COLS)}")
print(f"振動特徵數：{len(VIB_FEATURES)}")


特徵數：46
振動特徵數：43


In [3]:
# ============================================================
# 掃描檔案，只保留 scan 0–900
# ============================================================
all_files = sorted(
    glob.glob(os.path.join(DATA_DIR, "aligned_scan_*_7sensors_full.parquet")),
    key=lambda f: int(os.path.basename(f).split("_")[2])
)
print(f"資料夾共找到 {len(all_files)} 個 parquet 檔案")

# 只保留 scan_id 在 0–900 的檔案
filtered_files = [
    f for f in all_files
    if SCAN_MIN <= int(os.path.basename(f).split("_")[2]) <= SCAN_MAX
]
print(f"篩選後（scan {SCAN_MIN}~{SCAN_MAX}）：{len(filtered_files)} 個檔案")
print(f"第一個：{os.path.basename(filtered_files[0])}")
print(f"最後一個：{os.path.basename(filtered_files[-1])}")

# 7/3 切分
n_train     = int(len(filtered_files) * TRAIN_RATIO)
train_files = filtered_files[:n_train]
test_files  = filtered_files[n_train:]
print(f"\n訓練集：{len(train_files)} 個檔案（scan {SCAN_MIN} ~ {int(os.path.basename(train_files[-1]).split('_')[2])}）")
print(f"測試集：{len(test_files)} 個檔案（scan {int(os.path.basename(test_files[0]).split('_')[2])} ~ {SCAN_MAX}）")


資料夾共找到 901 個 parquet 檔案
篩選後（scan 0~900）：901 個檔案
第一個：aligned_scan_0_7sensors_full.parquet
最後一個：aligned_scan_900_7sensors_full.parquet

訓練集：630 個檔案（scan 0 ~ 629）
測試集：271 個檔案（scan 630 ~ 900）


In [4]:
# ============================================================
# 資料清理函式
# ============================================================
def load_and_clean(filepath, keep_cols):
    available = pd.read_parquet(filepath).columns.tolist()
    cols_to_read = [c for c in keep_cols if c in available]
    df = pd.read_parquet(filepath, columns=cols_to_read)
    # 移除第一筆波長點（1260 nm 異常讀值）
    df = df.sort_values("wavelength_nm").iloc[1:].reset_index(drop=True)
    # 移除 NaN
    df = df.dropna(subset=cols_to_read)
    # 移除 IL_Min 極端值
    df = df[(df["IL_Min"] >= -100) & (df["IL_Min"] <= 0)]
    return df


def process_files(file_list, keep_cols, output_path, desc="處理中"):
    chunks = [file_list[i:i+CHUNK_SIZE]
              for i in range(0, len(file_list), CHUNK_SIZE)]
    all_dfs    = []
    total_rows = 0

    for chunk_idx, chunk_files in enumerate(tqdm(chunks, desc=desc)):
        dfs = []
        for f in chunk_files:
            try:
                df = load_and_clean(f, keep_cols)
                dfs.append(df)
            except Exception as e:
                print(f"  [警告] {os.path.basename(f)}：{e}")

        if dfs:
            chunk_df = pd.concat(dfs, ignore_index=True)
            total_rows += len(chunk_df)
            all_dfs.append(chunk_df)
            del dfs, chunk_df
            gc.collect()

        print(f"  Chunk {chunk_idx+1}/{len(chunks)} 完成，累計 {total_rows:,} 筆")

    print(f"  合併中...")
    result = pd.concat(all_dfs, ignore_index=True)
    result.to_parquet(output_path, index=False, engine="pyarrow")
    print(f"  已儲存：{output_path}（{total_rows:,} 筆）")
    del all_dfs, result
    gc.collect()
    return total_rows

print("函式定義完成")


函式定義完成


In [5]:
# ============================================================
# Step 1：處理訓練集
# ============================================================
train_raw_path = os.path.join(OUTPUT_DIR, "train_raw.parquet")
n_train_rows = process_files(train_files, KEEP_COLS, train_raw_path, desc="訓練集")
print(f"\n訓練集總筆數：{n_train_rows:,}")


訓練集:   5%|▍         | 1/21 [00:01<00:31,  1.58s/it]

  Chunk 1/21 完成，累計 479,870 筆


訓練集:  10%|▉         | 2/21 [00:02<00:27,  1.44s/it]

  Chunk 2/21 完成，累計 959,697 筆


訓練集:  14%|█▍        | 3/21 [00:04<00:25,  1.40s/it]

...

訓練集:  95%|█████████▌| 20/21 [00:26<00:01,  1.32s/it]

  Chunk 20/21 完成，累計 9,596,570 筆


訓練集: 100%|██████████| 21/21 [00:27<00:00,  1.32s/it]

  Chunk 21/21 完成，累計 10,076,484 筆
  合併中...


  已儲存：C:\Users\peggy\Desktop\深度學習\processed_data\train_raw.parquet（10,076,484 筆）

訓練集總筆數：10,076,484


In [6]:
# ============================================================
# Step 2：處理測試集
# ============================================================
test_raw_path = os.path.join(OUTPUT_DIR, "test_raw.parquet")
n_test_rows = process_files(test_files, KEEP_COLS, test_raw_path, desc="測試集")
print(f"\n測試集總筆數：{n_test_rows:,}")


測試集:  10%|█         | 1/10 [00:01<00:11,  1.31s/it]

  Chunk 1/10 完成，累計 479,837 筆


測試集:  20%|██        | 2/10 [00:02<00:10,  1.25s/it]

  Chunk 2/10 完成，累計 959,658 筆


測試集:  30%|███       | 3/10 [00:03<00:08,  1.23s/it]

...

測試集:  80%|████████  | 8/10 [00:09<00:02,  1.20s/it]

  Chunk 8/10 完成，累計 3,838,255 筆


測試集: 100%|██████████| 10/10 [00:11<00:00,  1.10s/it]

  Chunk 9/10 完成，累計 4,318,056 筆
  Chunk 10/10 完成，累計 4,334,054 筆
  合併中...


  已儲存：C:\Users\peggy\Desktop\深度學習\processed_data\test_raw.parquet（4,334,054 筆）

測試集總筆數：4,334,054


In [7]:
# ============================================================
# Step 3：Fit StandardScaler（只用訓練集）
# ============================================================
print("讀取訓練集...")
train_df = pd.read_parquet(train_raw_path)
print(f"訓練集筆數：{len(train_df):,}")

scaler_X = StandardScaler()
scaler_y = StandardScaler()

print("Fit scaler...")
scaler_X.fit(train_df[FEATURE_COLS])
scaler_y.fit(train_df[[TARGET_COL]])

scaler_X_path = os.path.join(OUTPUT_DIR, "scaler_X.pkl")
scaler_y_path = os.path.join(OUTPUT_DIR, "scaler_y.pkl")
with open(scaler_X_path, "wb") as f: pickle.dump(scaler_X, f)
with open(scaler_y_path, "wb") as f: pickle.dump(scaler_y, f)
print(f"Scaler 已儲存")


讀取訓練集...
訓練集筆數：10,076,484
Fit scaler...
Scaler 已儲存


In [8]:
# ============================================================
# Step 4：標準化並儲存最終資料
# ============================================================
print("標準化訓練集...")
train_df[FEATURE_COLS] = scaler_X.transform(train_df[FEATURE_COLS])
train_df[[TARGET_COL]] = scaler_y.transform(train_df[[TARGET_COL]])
train_final_path = os.path.join(OUTPUT_DIR, "train_final.parquet")
train_df.to_parquet(train_final_path, index=False)
print(f"訓練集已儲存：{train_final_path}")
del train_df; gc.collect()

print("\n標準化測試集...")
test_df = pd.read_parquet(test_raw_path)
test_df[FEATURE_COLS] = scaler_X.transform(test_df[FEATURE_COLS])
test_df[[TARGET_COL]]  = scaler_y.transform(test_df[[TARGET_COL]])
test_final_path = os.path.join(OUTPUT_DIR, "test_final.parquet")
test_df.to_parquet(test_final_path, index=False)
print(f"測試集已儲存：{test_final_path}")
del test_df; gc.collect()

print("\n標準化完成！")


標準化訓練集...
訓練集已儲存：C:\Users\peggy\Desktop\深度學習\processed_data\train_final.parquet

標準化測試集...
測試集已儲存：C:\Users\peggy\Desktop\深度學習\processed_data\test_final.parquet

標準化完成！


In [9]:
# ============================================================
# Step 5：統計報告
# ============================================================
train_df = pd.read_parquet(train_final_path, columns=["scan_id","x_pos","y_pos","IL_Min"])
test_df  = pd.read_parquet(test_final_path,  columns=["scan_id","x_pos","y_pos","IL_Min"])

print("="*55)
print("資料前處理完成，統計報告")
print("="*55)

print(f"\n【訓練集】")
print(f"  總筆數：{len(train_df):,}")
print(f"  掃描數：{train_df['scan_id'].nunique()}")
print(f"  scan_id 範圍：{train_df['scan_id'].min()} ~ {train_df['scan_id'].max()}")
print(f"  座標分布：")
print(train_df.groupby(['x_pos','y_pos']).size().rename("筆數").to_string())

print(f"\n【測試集】")
print(f"  總筆數：{len(test_df):,}")
print(f"  掃描數：{test_df['scan_id'].nunique()}")
print(f"  scan_id 範圍：{test_df['scan_id'].min()} ~ {test_df['scan_id'].max()}")
print(f"  座標分布：")
print(test_df.groupby(['x_pos','y_pos']).size().rename("筆數").to_string())

print(f"\n【特徵資訊】")
print(f"  特徵數：{len(FEATURE_COLS)}")
print(f"  振動特徵數：{len(VIB_FEATURES)}")

# 儲存 metadata
meta = {
    "feature_cols":     FEATURE_COLS,
    "target_col":       TARGET_COL,
    "vib_features":     VIB_FEATURES,
    "train_ratio":      TRAIN_RATIO,
    "scan_range":       (SCAN_MIN, SCAN_MAX),
    "n_train_scans":    train_df['scan_id'].nunique(),
    "n_test_scans":     test_df['scan_id'].nunique(),
    "n_train_rows":     len(train_df),
    "n_test_rows":      len(test_df),
    "train_final_path": train_final_path,
    "test_final_path":  test_final_path,
    "scaler_X_path":    scaler_X_path,
    "scaler_y_path":    scaler_y_path,
}
meta_path = os.path.join(OUTPUT_DIR, "data_meta.pkl")
with open(meta_path, "wb") as f: pickle.dump(meta, f)
print(f"\n  Metadata：{meta_path}")
print("\n=== 全部完成 ===")


資料前處理完成，統計報告

【訓練集】
  總筆數：10,076,484
  掃描數：630
  scan_id 範圍：0 ~ 629
  座標分布：
x_pos      y_pos    
-1.552289   0.793414     959629
-0.871982  -0.676674    1119672
            0.793414     959606
-0.531829  -1.411718    1119693
            1.528458     959647
 0.148478   0.793414     943594
 0.488631  -1.411718    1119634
 0.828784   0.058370     959721
 1.168937  -0.676674     959631
 1.849244   0.793414     975657

【測試集】
  總筆數：4,334,054
  掃描數：271
  scan_id 範圍：630 ~ 900
  座標分布：
x_pos      y_pos    
-1.552289   0.793414    479855
-0.871982  -0.676674    319785
            0.793414    479771
-0.531829  -1.411718    319751
            1.528458    479781
 0.148478   0.793414    479795
 0.488631  -1.411718    335833
 0.828784   0.058370    479815
 1.168937  -0.676674    479811
 1.849244   0.793414    479857

【特徵資訊】
  特徵數：46
  振動特徵數：43

  Metadata：C:\Users\peggy\Desktop\深度學習\processed_data\data_meta.pkl

=== 全部完成 ===
